In [1]:
import torch
from torch import nn
import math
import copy
import numpy as np
import sentencepiece as spm
from tqdm import tqdm

import torch.nn.functional as F
from dataclasses import dataclass

from pathlib import Path
import urllib.request
import gzip
import shutil

from torchtext.data import Field, Example, Dataset
import sacrebleu

seed=42
torch.manual_seed(seed)
np.random.seed(seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

cuda


In [2]:
BASE = "https://raw.githubusercontent.com/multi30k/dataset/master/data/task1/raw"
FILES = [
    "train.de.gz", "train.en.gz",
    "val.de.gz", "val.en.gz",
    "test_2016_flickr.de.gz", "test_2016_flickr.en.gz",
]

out = Path("multi30k_de_en")
out.mkdir(exist_ok=True)

for name in FILES:
    gz_path = out / name
    txt_path = out / name.replace("test_2016_flickr", "test").replace(".gz", "")
    url = f"{BASE}/{name}"
    print(f"Downloading {url}")
    urllib.request.urlretrieve(url, gz_path)
    with gzip.open(gz_path, "rb") as f_in, open(txt_path, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

print("Done. Files created:")
for p in sorted(out.glob("*.de")) + sorted(out.glob("*.en")):
    print(p)

Done. Files created:
multi30k_de_en\test.de
multi30k_de_en\train.de
multi30k_de_en\val.de
multi30k_de_en\test.en
multi30k_de_en\train.en
multi30k_de_en\val.en


In [3]:
SRC = Field(
    tokenize="spacy",
    tokenizer_language="de_core_news_sm",
    init_token="<sos>",
    eos_token="<eos>",
    lower=True
)

TRG = Field(
    tokenize="spacy",
    tokenizer_language="en_core_web_sm",
    init_token="<sos>",
    eos_token="<eos>",
    lower=True
)


def load_translation_dataset(src_path, trg_path, src_field, trg_field):
    fields = [("src", src_field), ("trg", trg_field)]
    examples = []

    with open(src_path, encoding="utf-8") as src_file, open(trg_path, encoding="utf-8") as trg_file:
        for src_line, trg_line in zip(src_file, trg_file):
            src_line = src_line.strip()
            trg_line = trg_line.strip()

            if src_line and trg_line:
                example = Example.fromlist(
                    [src_line, trg_line],
                    fields
                )
                examples.append(example)

    return Dataset(examples, fields)


train_data = load_translation_dataset(
    "multi30k_de_en/train.de",
    "multi30k_de_en/train.en",
    SRC,
    TRG
)

valid_data = load_translation_dataset(
    "multi30k_de_en/val.de",
    "multi30k_de_en/val.en",
    SRC,
    TRG
)

test_data = load_translation_dataset(
    "multi30k_de_en/test.de",
    "multi30k_de_en/test.en",
    SRC,
    TRG
)

In [4]:
print(vars(train_data.fields['src'])) #type:ignore

{'sequential': True, 'use_vocab': True, 'init_token': '<sos>', 'eos_token': '<eos>', 'unk_token': '<unk>', 'fix_length': None, 'dtype': torch.int64, 'preprocessing': None, 'postprocessing': None, 'lower': True, 'tokenizer_args': ('spacy', 'de_core_news_sm'), 'tokenize': functools.partial(<function _spacy_tokenize at 0x000002B94A7F36A0>, spacy=<spacy.lang.de.German object at 0x000002B9023B96A0>), 'include_lengths': False, 'batch_first': False, 'pad_token': '<pad>', 'pad_first': False, 'truncate_first': False, 'stop_words': None, 'is_target': False}


# **German and English Tokenizers using SentencePiece**

In [ ]:
en_vocab_size = 8200
de_vocab_size = 10000
vocab_sizes = {"en": en_vocab_size, "de": de_vocab_size}

# Retrain with PAD token
spm.SentencePieceTrainer.Train(
    '--input=multi30k_de_en/train.de '
    '--model_prefix=Multi30k_de '
    '--vocab_size=10000 '
    '--user_defined_symbols=<pad>'
)

spm.SentencePieceTrainer.Train(
    '--input=multi30k_de_en/train.en '
    '--model_prefix=Multi30k_en '
    '--vocab_size=8200 '
    '--user_defined_symbols=<pad>'
)

# Load retrained models
de_sp = spm.SentencePieceProcessor()
de_sp.Load('Multi30k_de.model')
en_sp = spm.SentencePieceProcessor()
en_sp.Load('Multi30k_en.model')

UNK, BOS, EOS, PAD = 0, 1, 2, 3

# encode: text => id
print(en_sp.EncodeAsPieces('This is a test'))
print(en_sp.EncodeAsIds('This is a test'))

# decode: id => text
print(en_sp.DecodePieces(['▁This', '▁is', '▁a', '▁t', 'est']))
print(en_sp.DecodeIds([302, 258, 10, 4, 2395]))


tokenizers = {"en": en_sp.EncodeAsIds, "de": de_sp.EncodeAsIds}
detokenizers = {"en": en_sp.DecodeIds, "de": de_sp.DecodeIds}

['▁Th', 'is', '▁is', '▁a', '▁test']
[302, 258, 10, 4, 2395]
▁This is a test
This is a test


In [6]:
print([en_sp.id_to_piece([c]) for c in range(10)]) #type:ignore
print([de_sp.id_to_piece([c]) for c in range(10)]) #type:ignore

print(valid_data[0].trg)

UNK, BOS, EOS, PAD = 0, 1, 2, 3

[['<unk>'], ['<s>'], ['</s>'], ['<pad>'], ['▁a'], ['.'], ['▁A'], ['▁in'], ['▁the'], ['▁on']]
[['<unk>'], ['<s>'], ['</s>'], ['<pad>'], ['.'], ['▁eine'], ['▁Ein'], ['m'], ['▁in'], ['▁mit']]
['a', 'group', 'of', 'men', 'are', 'loading', 'cotton', 'onto', 'a', 'truck']


In [7]:
train_set = [
    (" ".join(ex.src), " ".join(ex.trg))
    for ex in train_data
    if len(ex.src) > 0 and len(ex.trg) > 0
]

valid_set = [
    (" ".join(ex.src), " ".join(ex.trg)) 
    for ex in valid_data
    if len(ex.src) > 0 and len(ex.trg) > 0
]

test_set = [
    (" ".join(ex.src), " ".join(ex.trg))
    for ex in test_data
    if len(ex.src) > 0 and len(ex.trg) > 0
]

print(len(train_set), len(valid_set), len(test_set))
for i in range(10):
   print(train_set[i])

29000 1014 1000
('zwei junge weiße männer sind im freien in der nähe vieler büsche .', 'two young , white males are outside near many bushes .')
('mehrere männer mit schutzhelmen bedienen ein antriebsradsystem .', 'several men in hard hats are operating a giant pulley system .')
('ein kleines mädchen klettert in ein spielhaus aus holz .', 'a little girl climbing into a wooden playhouse .')
('ein mann in einem blauen hemd steht auf einer leiter und putzt ein fenster .', 'a man in a blue shirt is standing on a ladder cleaning a window .')
('zwei männer stehen am herd und bereiten essen zu .', 'two men are at the stove preparing food .')
('ein mann in grün hält eine gitarre , während der andere mann sein hemd ansieht .', 'a man in green holds a guitar while the other man observes his shirt .')
('ein mann lächelt einen ausgestopften löwen an .', 'a man is smiling at a stuffed lion')
('ein schickes mädchen spricht mit dem handy während sie langsam die straße entlangschwebt .', 'a trendy gir

# SwiGLU

SwiGLU is a gated feed-forward layer used in many modern LLMs, it uses a **Swish** (or SiLU, beta=1) function, which preserves gradient flow and prevents dead neurons. It provides extra flexibility to strain on certain features while supressing others.

Refer : https://arxiv.org/pdf/2002.05202

In [8]:
"""
    SwiGLU(x, W, V) = [(xW + b) ⊙ σ(xW + b)] ⊙ (xV + c)
    FFN_SwiGLU(x) = SwiGLU(x, W, V) · W_out + b_out

    W, V : demb x dff
    W_out : dff x demb
    b,c : dff
    b_out : demb
    ⊙ - element-wise multiplication (Hadamard product)
"""

class SwiGLU(nn.Module):
    def __init__(self, demb):
        super().__init__()
        dff = int(demb * 8 / 3)
        self.w_up = nn.Linear(demb, dff)
        self.w_gate = nn.Linear(demb, dff)
        self.w_down = nn.Linear(dff, demb)

        # often kaiming works best in cases of swish or silu, though we are using sigmoid so we have to experiment to choose betweeen kaiming and xavier initialization

        nn.init.kaiming_uniform_(self.w_gate.weight, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.w_up.weight, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.w_down.weight, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
            return self.w_down(F.silu(self.w_up(x)) * self.w_gate(x))

# **Multi-Head Attention**

In [ ]:
def clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

def attention(q, k, v, mask=None, dropout=None):
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == True, -1e9)
    p_attn = scores.softmax(dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, v)

class MultiheadAttention(nn.Module):
    def __init__(self, demb, h, dropout=0.1):
        super().__init__()
        assert demb % h == 0
        self.d_k = demb // h
        self.h = h
        self.demb = demb
        self.linears = clones(nn.Linear(demb, demb), 4)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        nbatches = query.size(0)

        query, key, value = [linear(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2) for linear, x in zip(self.linears, (query, key, value))]
        """ 
            this results in query, key, value of shape (batch_size, num_heads, seq_len, d_k) 
            taking dv=dk=demb//h
        """

        """
            contiguous function maintains continuty in memory space even after transpose, or else it might show an error with view function is used to reshape the tensor to the desired shape directly from memory
        """
        x = attention(query, key, value, mask=mask, dropout=self.dropout) #type:ignore
        x = x.transpose(1, 2).contiguous().view(nbatches, -1, self.demb)
        return self.linears[-1](x)

FOR THE ENCODER-DECODER STRUCTURE WE USED THE ORIGINAL STRUCTURE DETAILED IN TEH PAPER (ATTENTION IS ALL YOU NEED), INSTEAD OF LayerNorm WE USED RMSNorm, (though we could have implemented sinusoidal position encoding layer, we let the model learn the positonal embeddings as well, as they both performed nearly the same, also mentioned in paper) 

In [10]:
class EncoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn = MultiheadAttention(config.demb, config.h, config.dropout)
        """
            the older tranformers used ff networks with linear layers and ReLU activation, but newer ones use SwiGLU which is a gated activation function that can capture more complex relationships in the data
        """
        self.ff = SwiGLU(config.demb)
        self.dropout = nn.Dropout(p=config.dropout)
        self.norm1 = nn.RMSNorm(config.demb)
        self.norm2 = nn.RMSNorm(config.demb)

    def forward(self, x, mask=None):
        attn_output = self.attn.forward(x, x, x, mask=mask)
        x = x + self.dropout(attn_output)
        x = self.norm1(x)

        ff_output = self.ff.forward(x)
        x = x + self.dropout(ff_output)
        x = self.norm2(x)

        return x

In [11]:
class Encoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.d_embed = config.demb
        self.tok_embed = nn.Embedding(config.encoder_vocab_size, config.demb) 
        self.pos_embed = nn.Parameter(torch.zeros(1, config.max_seq_len, config.demb).to(DEVICE)) 
        self.encoder_blocks = nn.ModuleList([EncoderBlock(config) for _ in range(config.N_encoder)])
        self.dropout = nn.Dropout(config.dropout)
        self.norm = nn.RMSNorm(config.demb)

    def forward(self, x, mask=None):
        x = self.tok_embed(x)
        x_pos = self.pos_embed[:, :x.size(1), :]
        x = self.dropout(x + x_pos)
        for layer in self.encoder_blocks:
            x = layer(x, mask)
        return self.norm(x)

In [12]:
class DecoderBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.self_attn = MultiheadAttention(config.demb, config.h, config.dropout)
        self.cross_attn = MultiheadAttention(config.demb, config.h, config.dropout)
        self.ff = SwiGLU(config.demb)
        self.dropout = nn.Dropout(p=config.dropout)
        self.norm1 = nn.RMSNorm(config.demb)
        self.norm2 = nn.RMSNorm(config.demb)
        self.norm3 = nn.RMSNorm(config.demb)

    def forward(self, x, enc_output, src_pad_mask=None, tgt_pad_mask=None, casual_mask=None):
        if tgt_pad_mask is not None and casual_mask is not None:
            combined_mask = tgt_pad_mask | casual_mask
            self_attn_output = self.self_attn.forward(x, x, x, mask=combined_mask)
        else:
            self_attn_output = self.self_attn.forward(x, x, x, mask=tgt_pad_mask)
        x = x + self.dropout(self_attn_output)
        x = self.norm1(x)

        cross_attn_output = self.cross_attn.forward(x, enc_output, enc_output, mask=src_pad_mask)
        x = x + self.dropout(cross_attn_output)
        x = self.norm2(x)

        ff_output = self.ff.forward(x)
        x = x + self.dropout(ff_output)
        x = self.norm3(x)

        return x

In [13]:
class Decoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.tok_embed = nn.Embedding(config.decoder_vocab_size, config.demb)
        self.pos_embed = nn.Parameter(torch.zeros(1, config.max_seq_len, config.demb).to(DEVICE))
        self.layers = clones(DecoderBlock(config), config.N_decoder)
        self.norm = nn.RMSNorm(config.demb)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None, casual_mask=None):
        x = self.tok_embed(x)
        x_pos = self.pos_embed[:, :x.size(1), :]
        x = self.dropout(x + x_pos)
        for layer in self.layers:
            x = layer(x, enc_output, src_pad_mask=src_mask, tgt_pad_mask=tgt_mask, casual_mask=casual_mask)
        return x

In [14]:
@dataclass
class ModelConfig:
    encoder_vocab_size: int
    decoder_vocab_size: int
    demb: int
    h: int
    N_encoder: int
    N_decoder: int
    max_seq_len: int
    dropout: float
    
class Transformer(nn.Module):
    def __init__(self, encoder, decoder, config):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.proj = nn.Linear(config.demb, config.decoder_vocab_size)

    def forward(self, src, src_mask, trg, trg_pad_mask, casual_mask=None):
        enc_output = self.encoder(src, src_mask)
        dec_output = self.decoder(trg, enc_output, src_mask, trg_pad_mask, casual_mask)
        return self.proj(dec_output)

def make_model(config):
    model = Transformer(Encoder(config), Decoder(config), config).to(DEVICE)

    # initialize model parameters
    # it seems that this initialization is very important!
    for p in model.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    return model

# Preparing the tokenised dataset for training using dataloaders

In [15]:
max_seq_len = 50
def tokenize_dataset(dataset):
    return [
        (
            torch.tensor(
                [BOS] + tokenizers["de"](src_text)[:max_seq_len - 2] + [EOS],  # ✓ German for German
                dtype=torch.long
            ),
            torch.tensor(
                [BOS] + tokenizers["en"](trg_text)[:max_seq_len - 2] + [EOS],  # ✓ English for English
                dtype=torch.long
            )
        )
        for src_text, trg_text in dataset
    ]

# Retokenize all datasets
train_tokenized = tokenize_dataset(train_set)
valid_tokenized = tokenize_dataset(valid_set)
test_tokenized = tokenize_dataset(test_set)


class TranslationDataset(torch.utils.data.Dataset):
    'create a dataset for torch.utils.data.DataLoader() '
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def pad_sequence(batch):
    'collate function for padding sentences such that all \
    the sentences in the batch have the same length'
    src_seqs  = [src for src, trg in batch]
    trg_seqs  = [trg for src, trg in batch]
    src_padded = torch.nn.utils.rnn.pad_sequence(src_seqs,
                                batch_first=True, padding_value = PAD)
    trg_padded = torch.nn.utils.rnn.pad_sequence(trg_seqs,
                                batch_first=True, padding_value = PAD)
    return src_padded, trg_padded

print(en_sp.DecodeIds(train_tokenized[0][1].tolist()))

two young , white males are outside near many bushes .


In [16]:
batch_size = 128

class Dataloaders:
    'Dataloaders contains train_loader, test_loader and valid_loader for training and evaluation '
    def __init__(self):
        self.train_dataset = TranslationDataset(train_tokenized)
        self.valid_dataset = TranslationDataset(valid_tokenized)
        self.test_dataset  = TranslationDataset(test_tokenized)
        
        # each batch returned by dataloader will be padded such that all the texts in
        # that batch have the same length as the longest text in that batch
        self.train_loader = torch.utils.data.DataLoader(self.train_dataset, batch_size=batch_size,
                                                shuffle=True, collate_fn = pad_sequence)
        self.valid_loader = torch.utils.data.DataLoader(self.valid_dataset, batch_size=batch_size,
                                                shuffle=True, collate_fn = pad_sequence)
        self.test_loader = torch.utils.data.DataLoader(self.test_dataset, batch_size=batch_size,
                                                shuffle=True, collate_fn = pad_sequence)

# Creating the casual mask and implementing training script

`make_batch_inputs()`: This is the crux of passing our data intot the model, it takes a pair consisiting of the german language to be translated along with one of it's translation, 
src = x
trg_in: It is the sequence of tokens used to train the decoder model to predict the next set of sequence tokens (trg_out)
trg_out: the expected final output of the decoder model for the model

then we make a padding mask to make sure the model does not give any attention to it

We are using cross-entropy loss along with adam optimizer, I also added a NaN runtime error. 

In [ ]:

def create_causal_mask(seq_length, device):
    """
    Create a causal attention mask for the decoder.
    Prevents attention to future positions.
    
    Returns:
        mask of shape (1, 1, seq_length, seq_length) where True = mask out
    """
    mask = torch.triu(torch.ones(seq_length, seq_length, device=device), diagonal=1).bool()
    return mask.unsqueeze(0).unsqueeze(0)

def make_batch_input(x, y):
    src = x.to(DEVICE)

    trg_in = y[:, :-1].to(DEVICE)

    trg_out = y[:, 1:].contiguous().view(-1).to(DEVICE)

    src_pad_mask = (src == PAD).view(src.size(0), 1, 1, src.size(-1))

    trg_pad_mask = (trg_in == PAD).view(
        trg_in.size(0), 1, 1, trg_in.size(-1)
    )
    
    causal_mask = create_causal_mask(trg_in.size(1), DEVICE)

    return src, trg_in, trg_out, src_pad_mask, trg_pad_mask, causal_mask


def train_epoch(model, dataloaders, optim, scheduler, loss_fn):
    model.train()

    grad_norm_clip = 1.0
    losses = []

    num_batches = len(dataloaders.train_loader)

    pbar = tqdm(
        enumerate(dataloaders.train_loader),
        total=num_batches
    )

    for idx, (x, y) in pbar:
        optim.zero_grad()

        src, trg_in, trg_out, src_pad_mask, trg_pad_mask, casual_mask = make_batch_input(x, y)

        pred = model(src, src_pad_mask, trg_in, trg_pad_mask, casual_mask)

        # check for NaNs in model outputs
        if torch.isnan(pred).any():
            print("NaN detected in model outputs at batch", idx)
            print("src dtype/device:", src.dtype, src.device)
            for name, p in model.named_parameters():
                if torch.isnan(p).any():
                    print("NaN in parameter:", name)
                    break
            raise RuntimeError("NaN in model outputs")

        pred = pred.view(-1, pred.size(-1))

        loss = loss_fn(pred, trg_out)

        # Diagnostic: check for NaN/Inf loss
        if torch.isnan(loss) or torch.isinf(loss):
            print("NaN/Inf loss at batch", idx)
            torch.autograd.set_detect_anomaly(True)
            raise RuntimeError("NaN/Inf loss")

        loss.backward()

        # Diagnostic: check for NaNs in grads
        for name, p in model.named_parameters():
            if p.grad is not None and torch.isnan(p.grad).any():
                print("NaN in gradient for parameter:", name)
                raise RuntimeError("NaN in gradients")

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            grad_norm_clip
        )

        optim.step()
        scheduler.step()

        losses.append(loss.item())

        if idx > 0 and idx % 50 == 0:
            pbar.set_description(
                f"train loss={loss.item():.3f}, "
                f"lr={scheduler.get_last_lr()[0]:.5f}"
            )

    return np.mean(losses)


def validate(model, dataloader):
    """
    Compute the validation loss.
    """

    model.eval()
    losses = []

    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)

    with torch.no_grad():
        for x, y in dataloader:
            src, trg_in, trg_out, src_pad_mask, trg_pad_mask, casual_mask = make_batch_input(x, y)

            pred = model.forward(src, src_pad_mask, trg_in, trg_pad_mask, casual_mask)

            pred = pred.view(-1, pred.size(-1))

            loss = loss_fn(pred, trg_out)

            losses.append(loss.item())

    return np.mean(losses)

def train(model, dataloaders, epochs, optim, scheduler, warmup_steps, loss_fn):
    global early_stop_count

    best_valid_loss = float("inf")

    for ep in range(epochs):
        train_loss = train_epoch(model, dataloaders, optim, scheduler, loss_fn)

        valid_loss = validate(model, dataloaders.valid_loader)

        print(
            f"ep: {ep}: "
            f"train_loss={train_loss:.5f}, "
            f"valid_loss={valid_loss:.5f}"
        )

        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss

        else:
            if scheduler.last_epoch > 2 * warmup_steps:
                early_stop_count -= 1

                if early_stop_count <= 0:
                    return train_loss, valid_loss

    return train_loss, valid_loss

# INFERENCE CODE

In [ ]:
def translate(model, x):
    '''translate source sentences into the target language, without looking at the answer'''
    with torch.no_grad():
        dB = x.size(0)
        y = torch.tensor([[BOS]*dB], dtype=torch.long, device=DEVICE).view(dB, 1)
        x_pad_mask = (x == PAD).view(x.size(0), 1, 1, x.size(-1))
        
        for _ in range(max_seq_len):
            y_pad_mask = (y == PAD).view(y.size(0), 1, 1, y.size(-1))
            causal_mask = create_causal_mask(y.size(1), DEVICE)
            
            logits = model(x, x_pad_mask, y, y_pad_mask, causal_mask)
            
            last_output = logits.argmax(-1)[:, -1]
            last_output = last_output.view(dB, 1)
            y = torch.cat((y, last_output), 1)
    return y

def remove_pad(sent):
    '''truncate the sentence if BOS is in it,
     otherwise simply remove the padding tokens at the end'''
    if sent.count(EOS)>0:
      sent = sent[0:sent.index(EOS)+1]
    while sent and sent[-1] == PAD:
            sent = sent[:-1]
    return sent

def decode_sentence(detokenizer, sentence_ids):
    'convert a tokenized sentence (a list of numbers) to a literal string'
    if not isinstance(sentence_ids, list):
        sentence_ids = sentence_ids.tolist()
    sentence_ids = remove_pad(sentence_ids)
    return detokenizer(sentence_ids).replace("", "")\
           .replace("", "").strip().replace(" .", ".")

def evaluate(model, dataloader, num_batch=None):
    '''evaluate the model, and compute the BLEU score'''
    model.eval()
    refs, cans, bleus = [], [], []
    with torch.no_grad():
        for idx, (x, y) in enumerate(dataloader):
            src, trg_in, trg_out, src_pad_mask, trg_pad_mask, causal_mask = make_batch_input(x, y)
            translation = translate(model, src)
            trg_out = trg_out.view(x.size(0), -1)
            refs = refs + [decode_sentence(detokenizers["en"], trg_out[i]) for i in range(len(src))]
            cans = cans + [decode_sentence(detokenizers["en"], translation[i]) for i in range(len(src))] 
            if num_batch and idx >= num_batch:
                break
        print(min([len(x) for x in refs]))
        bleus.append(sacrebleu.corpus_bleu(cans, [refs]).score)
        # print some examples
        for i in range(3):
            # FIX: Correct detokenizers (German for source, English for target/predictions)
            print(f'src:  {decode_sentence(detokenizers["de"], src[i])}')
            print(f'trg:  {decode_sentence(detokenizers["en"], trg_out[i])}')
            print(f'pred: {decode_sentence(detokenizers["en"], translation[i])}')
        return np.mean(bleus)

# Training


In [19]:
config = ModelConfig(
    encoder_vocab_size = vocab_sizes["de"],      # ✓ German (10000)
    decoder_vocab_size = vocab_sizes["en"],      # ✓ English (8200)
    demb=512,
    h=8,
    N_encoder=3, 
    N_decoder=3, 
    max_seq_len=max_seq_len,
    dropout=0.1
)

def make_lr_fn(warmup_steps, demb):
    """Return a callable lr_lambda(step) for LambdaLR using the """
    """Transformer warmup schedule: demb^{-0.5} * min((step+1)^{-0.5}, (step+1)*warmup^{-1.5})"""
    def lr_lambda(step: int) -> float:
        step = max(step, 0)
        return demb ** (-0.5) * min((step + 1) ** (-0.5), (step + 1) * (warmup_steps ** -1.5))
    return lr_lambda


data_loaders = Dataloaders()
train_size = len(data_loaders.train_loader)*batch_size
model = make_model(config)
model_size = sum([p.numel() for p in model.parameters()])
print(f'model_size: {model_size}, train_set_size: {train_size}')
warmup_steps = 3*len(data_loaders.train_loader)
# lr first increases in the warmup steps, and then decreases
lr_fn = make_lr_fn(warmup_steps, config.demb)
# Use a stable initial optimizer LR to avoid exploding gradients
optimizer = torch.optim.Adam(model.parameters(), lr=0.5, betas=(0.9, 0.98), eps=1e-9)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_fn)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)
early_stop_count = 5
train_loss, valid_loss = train(model, data_loaders, epochs=12, optim=optimizer, scheduler=scheduler, warmup_steps=warmup_steps, loss_fn=loss_fn)
test_loss  = validate(model, data_loaders.test_loader)

model_size: 35639812, train_set_size: 29056


train loss=3.700, lr=0.00025: 100%|██████████| 227/227 [03:11<00:00,  1.19it/s]


ep: 0: train_loss=5.33003, valid_loss=3.48984


train loss=2.760, lr=0.00053: 100%|██████████| 227/227 [00:41<00:00,  5.50it/s]


ep: 1: train_loss=2.97678, valid_loss=2.42626


...

ep: 9: train_loss=0.60378, valid_loss=1.66688


train loss=0.542, lr=0.00044: 100%|██████████| 227/227 [00:51<00:00,  4.42it/s]


ep: 10: train_loss=0.50859, valid_loss=1.70958


train loss=0.467, lr=0.00043: 100%|██████████| 227/227 [00:51<00:00,  4.40it/s]


ep: 11: train_loss=0.43281, valid_loss=1.78191


In [20]:
def translate_this_sentence(text: str):
    '''translate German source to English target'''
    tokens = [BOS] + tokenizers["de"](text) + [EOS]
    
    # Pad/truncate to max_seq_len
    if len(tokens) > max_seq_len:
        tokens = tokens[:max_seq_len]
    else:
        tokens = tokens + [PAD] * (max_seq_len - len(tokens))
    
    # Create batch tensor (1, seq_len)
    input_tensor = torch.tensor([tokens], dtype=torch.long).to(DEVICE)
    
    with torch.no_grad():
        output = translate(model, input_tensor)
    
    return decode_sentence(detokenizers["en"], output[0])


# Test
result = translate_this_sentence("Eine Gruppe von Menschen steht vor einem Iglu.")
print(result)

ethnic students are standing in front of an extity of bread.


In [ ]:
# Step 1: Fix tokenize_dataset (swap tokenizers)
def tokenize_dataset(dataset):
    return [
        (
            torch.tensor(
                [BOS] + tokenizers["de"](src_text)[:max_seq_len - 2] + [EOS],  # ✓ German tokenizer
                dtype=torch.long
            ),
            torch.tensor(
                [BOS] + tokenizers["en"](trg_text)[:max_seq_len - 2] + [EOS],  # ✓ English tokenizer
                dtype=torch.long
            )
        )
        for src_text, trg_text in dataset
    ]

# Retokenize
train_tokenized = tokenize_dataset(train_set)
valid_tokenized = tokenize_dataset(valid_set)
test_tokenized = tokenize_dataset(test_set)

# Step 2: Fix vocab sizes in ModelConfig
config = ModelConfig(
    encoder_vocab_size = vocab_sizes["de"],      # ✓ German (10000)
    decoder_vocab_size = vocab_sizes["en"],      # ✓ English (8200)
    demb=512,
    h=8,
    N_encoder=3, 
    N_decoder=3, 
    max_seq_len=max_seq_len,
    dropout=0.1
)

# Step 3: Create FRESH model with correct config
model = make_model(config)
model_size = sum([p.numel() for p in model.parameters()])
print(f'Fresh model size: {model_size}')

# Step 4: Recreate dataloaders with fixed tokenized data
data_loaders = Dataloaders()

# Step 5: Train fresh
optimizer = torch.optim.Adam(model.parameters(), lr=0.5, betas=(0.9, 0.98), eps=1e-9)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_fn)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)

print("Starting fresh training with ALL fixes...")
train_loss, valid_loss = train(
    model, 
    data_loaders, 
    epochs=20,
    optim=optimizer, 
    scheduler=scheduler, 
    warmup_steps=warmup_steps, 
    loss_fn=loss_fn
)

# Step 6: Test after retraining
print("\n" + "="*50)
print("Testing after retraining:")
print("="*50)
result = translate_this_sentence("Eine Gruppe von Menschen steht vor einem Iglu.")
print("Input:  Eine Gruppe von Menschen steht vor einem Iglu.")
print(f"Output: {result}")

Fresh model size: 35639812
Starting fresh training with ALL fixes...


train loss=3.785, lr=0.00025: 100%|██████████| 227/227 [00:50<00:00,  4.46it/s]


ep: 0: train_loss=5.29200, valid_loss=3.47961


train loss=2.440, lr=0.00053: 100%|██████████| 227/227 [00:53<00:00,  4.21it/s]


ep: 1: train_loss=2.96741, valid_loss=2.40778


...

ep: 6: train_loss=1.03179, valid_loss=1.60072


train loss=0.842, lr=0.00052: 100%|██████████| 227/227 [00:54<00:00,  4.16it/s]


ep: 7: train_loss=0.86496, valid_loss=1.59104


train loss=0.773, lr=0.00049: 100%|██████████| 227/227 [00:54<00:00,  4.17it/s]


ep: 8: train_loss=0.72549, valid_loss=1.62281

Testing after retraining:
Input:  Eine Gruppe von Menschen steht vor einem Iglu.
Output: dareiftr stands in front of an empty ready.


In [22]:
result = translate_this_sentence("Zwei Jungen tanzen auf Stangen .")
print(result)

during wovenonea everyone is dancing.


In [23]:
train_bleu = evaluate(model, data_loaders.train_loader, 20)

19
src:  ein kleines mädchen fährt in einem geschäft vorne im einkaufswagen.
trg:  a little girl is riding on the front of a shopping cart in a store.
pred: a little girl is riding in the front of store in a shopping cart.
src:  ein schwarzer hund und ein weißer hund spielen miteinander und einem grünen ball.
trg:  a black dog and a white dog play with each other and a green ball.
pred: a black dog and a white dog are playing together and one green ball.
src:  ein junge fängt einen fisch , während ein mädchen auf diesen fisch zeigt.
trg:  a boy caches a fish while a girl points at the fish.
pred: a boy is catching a fish while a girl points at the fish.


In [24]:
test_blue = evaluate(model, data_loaders.test_loader)

21
src:  ein angestellter reicht einer frau auf einem markt eine tüte , während sie auf eis gelegten fisch begutachtet.
trg:  an employee is handing a woman a bag while she is browsing through fish on ice at a street market.
pred: a grocery store is handing a woman in a market as she looks at fish.
src:  ein schwarzer junge sitzt im sand.
trg:  a black boy is sitting in the sand.
pred: a black boy is sitting in the sand.
src:  ein geschäftiger tag auf einem städtischen platz.
trg:  a busy day for citizens at a local city court.
pred: a busy day in a city square.
